In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

train = pd.read_csv('data/train_clean.csv', low_memory=False)
test  = pd.read_csv('data/test_clean.csv',  low_memory=False)

train['Date'] = pd.to_datetime(train['Date'])
test['Date']  = pd.to_datetime(test['Date'])

print("Train shape:", train.shape)
print("Test shape :", test.shape)

Train shape: (844338, 18)
Test shape : (41088, 17)


In [2]:
# Extract useful date parts from the Date column
def add_date_features(df):
    df['Year']        = df['Date'].dt.year
    df['Month']       = df['Date'].dt.month
    df['Day']         = df['Date'].dt.day
    df['WeekOfYear']  = df['Date'].dt.isocalendar().week.astype(int)
    df['Quarter']     = df['Date'].dt.quarter
    df['IsWeekend']   = (df['DayOfWeek'] >= 6).astype(int)
    df['IsMonthStart'] = df['Date'].dt.is_month_start.astype(int)
    df['IsMonthEnd']   = df['Date'].dt.is_month_end.astype(int)
    return df

train = add_date_features(train)
test  = add_date_features(test)

print("New date columns added:")
print(['Year','Month','Day','WeekOfYear','Quarter',
       'IsWeekend','IsMonthStart','IsMonthEnd'])

New date columns added:
['Year', 'Month', 'Day', 'WeekOfYear', 'Quarter', 'IsWeekend', 'IsMonthStart', 'IsMonthEnd']


In [3]:
# How many months has the nearest competitor been open?
def add_competition_age(df):
    df['CompetitionOpen'] = (
        12 * (df['Year'] - df['CompetitionOpenSinceYear']) +
        (df['Month'] - df['CompetitionOpenSinceMonth'])
    )
    # If no competitor data, set to 0
    df['CompetitionOpen'] = df['CompetitionOpen'].apply(
        lambda x: x if x > 0 else 0)
    return df

train = add_competition_age(train)
test  = add_competition_age(test)

print("CompetitionOpen sample:")
print(train['CompetitionOpen'].describe())

CompetitionOpen sample:
count    844338.000000
mean       7731.474911
std       11229.439988
min           0.000000
25%          29.000000
50%          91.000000
75%       24163.000000
max       24187.000000
Name: CompetitionOpen, dtype: float64


In [5]:
# Is Promo2 currently active for this store on this date?
month_map = {'Jan':1,'Feb':2,'Mar':3,'Apr':4,'May':5,'Jun':6,
             'Jul':7,'Aug':8,'Sep':9,'Sept':9,'Oct':10,'Nov':11,'Dec':12}

def is_promo2_active(row):
    if row['Promo2'] == 0:
        return 0
    if row['PromoInterval'] == 'None':
        return 0
    try:
        months = [month_map[m] for m in row['PromoInterval'].split(',')]
        if row['Month'] in months:
            return 1
        return 0
    except:
        return 0

train['IsPromo2Active'] = train.apply(is_promo2_active, axis=1)
test['IsPromo2Active']  = test.apply(is_promo2_active, axis=1)

print("Promo2 active distribution:")
print(train['IsPromo2Active'].value_counts())

Promo2 active distribution:
IsPromo2Active
0    699125
1    145213
Name: count, dtype: int64


In [6]:
# Encode each store's average sales behaviour
store_stats = train.groupby('Store')['Sales'].agg(
    StoreMeanSales   = 'mean',
    StoreMedianSales = 'median',
    StoreStdSales    = 'std'
).reset_index()

train = train.merge(store_stats, on='Store', how='left')
test  = test.merge(store_stats, on='Store', how='left')

print("Store stats added:")
print(train[['Store','StoreMeanSales','StoreMedianSales','StoreStdSales']].head())

Store stats added:
   Store  StoreMeanSales  StoreMedianSales  StoreStdSales
0      1     4759.096031            4647.0    1012.106393
1      2     4953.900510            4783.0    1610.149102
2      3     6942.568678            6619.0    2193.383804
3      4     9638.401786            9430.5    1936.031881
4      5     4676.274711            4616.0    1765.745628


In [7]:
# Convert text categories to numbers for the model
train['StoreType']   = train['StoreType'].map({'a':0,'b':1,'c':2,'d':3})
train['Assortment']  = train['Assortment'].map({'a':0,'b':1,'c':2})
train['StateHoliday'] = train['StateHoliday'].map(
    {'none':0,'public':1,'easter':2,'christmas':3})

test['StoreType']    = test['StoreType'].map({'a':0,'b':1,'c':2,'d':3})
test['Assortment']   = test['Assortment'].map({'a':0,'b':1,'c':2})
test['StateHoliday'] = test['StateHoliday'].map(
    {'none':0,'public':1,'easter':2,'christmas':3})

print("Encoding done!")
print(train[['StoreType','Assortment','StateHoliday']].head())

Encoding done!
   StoreType  Assortment  StateHoliday
0          2           0             0
1          0           0             0
2          0           0             0
3          2           2             0
4          0           0             0


In [8]:
# Save the fully featured datasets
train.to_csv('data/train_features.csv', index=False)
test.to_csv('data/test_features.csv', index=False)

print("Saved train_features.csv and test_features.csv!")
print("Train shape:", train.shape)
print("Test shape :", test.shape)
print("Total features:", train.shape[1])

Saved train_features.csv and test_features.csv!
Train shape: (844338, 31)
Test shape : (41088, 30)
Total features: 31
